# 州×商品カテゴリから販促候補を考える

## このNotebookの問い

PB・APはSPより平均注文額が高い。一方で注文数は少ない。各州で何が購入されているかを確認し、販促候補を絞る前に必要な情報を整理する。

ここで分かるのは過去の注文構成であり、広告を出した場合の追加売上や利益ではない。

**指標の定義**

- 州の平均注文額 = 州のGMV ÷ 州のユニークな注文数
- カテゴリ購入割合 = そのカテゴリを含む注文数 ÷ 州のユニークな注文数
- 州内カテゴリGMVシェア = そのカテゴリのGMV ÷ 州のGMV

1注文に複数カテゴリがある場合、カテゴリ購入割合の合計は100%を超えることがある。

In [3]:
from pathlib import Path
import sys

# Notebookをどの作業フォルダから開いてもsrc/を読み込めるようにする。
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src").is_dir() and (path / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.project_paths import DATA_DIR
from src.sales_metrics import state_totals as build_state_totals, category_state_metrics

df_baseline = pd.read_csv(DATA_DIR / "sales_baseline_by_month_state.csv")
df_category = pd.read_csv(DATA_DIR / "sales_by_category_state.csv")
target_states = ["SP", "PB", "AP"]

print(df_baseline.head())
print(df_baseline.shape)
print(df_category.head())
print(df_category.shape)

  purchase_month customer_state  order_count      gmv  total_freight  \
0        2016-09             SP            1   134.97           8.49   
1        2016-10             AL            1    49.99          26.61   
2        2016-10             BA            3   244.88          76.13   
3        2016-10             CE            6  1649.68         235.70   
4        2016-10             DF            6  1043.77         156.34   

   average_order_value  
0           134.970000  
1            49.990000  
2            81.626667  
3           274.946667  
4           173.961667  
(556, 6)
  customer_state       product_category  category_order_count  \
0             AC         sports_leisure                     9   
1             AC          watches_gifts                     4   
2             AC          health_beauty                     6   
3             AC        furniture_decor                     7   
4             AC  computers_accessories                     6   

   order_item_lin

In [ ]:
# 州全体の注文数は、カテゴリ件数ではなく月×州の基準表から作る。
state_totals = build_state_totals(df_baseline)
display(state_totals.loc[
    state_totals["customer_state"].isin(target_states),
    ["customer_state", "state_orders", "state_gmv", "average_order_value"],
].round(2))

In [ ]:
# 州の注文数を分母にしたカテゴリ注文率と、州GMV内の構成比を確認する。
category_comparison = category_state_metrics(df_category, df_baseline)
category_comparison = category_comparison.loc[
    category_comparison["customer_state"].isin(target_states)
]
top_categories = (
    category_comparison.sort_values(
        ["customer_state", "category_gmv"], ascending=[True, False]
    ).groupby("customer_state", group_keys=False).head(5)
)
display(top_categories[[
    "customer_state", "product_category", "category_order_count",
    "category_order_rate_pct", "category_gmv", "category_gmv_share_pct",
]].round(2))

## 結果を読む際の確認事項

1. PB・APの上位カテゴリは、何件の注文に支えられているか。少数の高額注文に偏っていないか。
2. SPと同じカテゴリでも、購入割合とGMVシェアはどう違うか。
3. この集計だけで需要の不足や広告の効果を断定しない。次に必要な情報は、期間別の安定性、商品・販売者の構成、広告費・利益に関する情報。

結果を確認してから、どのカテゴリを詳しく調べるかを決める。配送フォロー施策については、別途レビュー・注文時系列の分析条件を整理する。